# Batch processing with complete global metadata

The supplied Sheet1 has Raw File Name, Raw File Location, New File Name, Background Mode, Background File, Backgorund File Location, and ten field metadata columns.
`Field Note` replaces `user_notes`. The single-file notebook writes all acquisition,
processing-summary and field metadata into every public a1/2/3 NetCDF.

Choose `RUN_MODE` in section 1:
- `metadata_only`: update public NetCDF attributes and companion metadata for successful jobs in the existing ledger. Never run RAW processing. Missing ledger entries are reported.
- `process`: normal processing and resume behavior. Changing only field metadata refreshes attributes for a matching successful run. This code revision has a new fingerprint, so older runs can be reprocessed in this mode. Use metadata_only to upgrade older results.

Dates use YYYY-MM-DD and times use UTC HH:MM[:SS]. Blank optional fields remain missing. No location, altitude or sampling dates are inferred from the RAW.

New File Name is required for each job and has the form prefix.YYYYMMDD.HHMMSS. a1/b1/c1 are inserted before the date. Metadata-only mode also renames completed outputs and updates saved state paths. It never reruns RAW processing.

This notebook loads 01_process_single_raw_file_gap_guard.ipynb. metadata_only physically migrates existing successful public files to a1/b1/c1 attributes, renames date to data_acquisition_date and removes obsolete globals. New File Name remains an Excel naming input and diagnostic configuration value, not a public global attribute. Existing ledger keys are preserved.

This version loads the right-tail gap guard from 01. Use process mode for newly computed scientific results; metadata_only cannot change retained peaks. The processing-code fingerprint changes, so prior runs may be recomputed in process mode.


## 1. Set the working folder and path

Set `WORK_DIR` to the folder containing this notebook, and the single-file notebook. JOBS_XLSX defaults to the supplied workbook in Downloads. Set `CODE_DIR` to the folder containing `corems_topn_processing.py`. The formula database and reference mass list stay in **Section 2 of the single-file notebook**. The batch summary and progress ledger go into `WORK_DIR/named_output_reports`.
Set DEFAULT_OUTPUT_ROOT for generated runs. JOBS_SHEET defaults to Sheet1. The workbook is read at execution time, so save your corrected New File Name values before running.

ONLY_EXCEL_ROWS defaults to {5}, the affected EMSL60338_nanoDESI_GV_Neg_051323_F3_V2_01 sample in the current workbook. Verify the sample name in the preflight table before processing. Set ONLY_EXCEL_ROWS = None to process the full workbook.


In [8]:
from pathlib import Path

WORK_DIR = Path(r"C:\Users\nius206\Documents\ARM_nanodesi")
SingleFile_NOTEBOOK = WORK_DIR / "01_process_single_raw_file.ipynb"
JOBS_XLSX = Path(r"C:\Users\nius206\Documents\ARM_nanodesi\batch2\raw_wo_background.xlsx")
CODE_DIR = Path(r"C:\Users\nius206\Documents\ARM_nanodesi")
REPORT_DIR = WORK_DIR / "named_output_reports"

# Normally successful jobs with unchanged scientific inputs are skipped.
# Add Excel row numbers here to intentionally repeat an already completed RAW.
FORCE_RERUN_EXCEL_ROWS = set()  # e.g. {2, 5}
# For metadata_only on older results, set the original workbook identity and REPORT_DIR.
LEDGER_IDENTITY_XLSX = JOBS_XLSX
RUN_MODE = "process"  # metadata_only | process

JOBS_SHEET = "Sheet1"
# The supplied workbook has no output-location column.
DEFAULT_OUTPUT_ROOT = WORK_DIR / "batch2"

# Validate selected sample only.
# Set to None to process the whole manifest.
ONLY_EXCEL_ROWS = None


## 2. Load the single-file settings and functions

The batch loads 01 directly. Metadata support now lives in 01, so no string-injection patch is needed. The RAW-processing call in 01 is not executed here.


In [9]:
import ast
import copy
from dataclasses import asdict
from datetime import datetime, timezone
import hashlib
import importlib
import json
import os
import re
import sys
import traceback
import uuid

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from netCDF4 import Dataset
from sqlalchemy.engine import make_url

SingleFile_NOTEBOOK = SingleFile_NOTEBOOK.expanduser().resolve()
JOBS_XLSX = JOBS_XLSX.expanduser().resolve()
CODE_DIR = CODE_DIR.expanduser().resolve()
REPORT_DIR = REPORT_DIR.expanduser().resolve()
module_file = CODE_DIR / "corems_topn_processing.py"
for required in (SingleFile_NOTEBOOK, JOBS_XLSX, module_file):
    if not required.is_file():
        raise FileNotFoundError(f"Set the correct path in Section 1: {required}")

if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))
if "corems_topn_processing" in sys.modules:
    loaded = Path(sys.modules["corems_topn_processing"].__file__).resolve()
    if loaded != module_file:
        raise RuntimeError(f"Kernel already loaded a different CoreMS module: {loaded}. Restart the kernel.")
    topn = importlib.reload(sys.modules["corems_topn_processing"])
else:
    import corems_topn_processing as topn
from corems_topn_processing import CoreMSConfig, BackgroundConfig

source_notebook = json.loads(SingleFile_NOTEBOOK.read_text(encoding="utf-8"))
source_cells = ["".join(cell["source"]) if isinstance(cell["source"], list) else cell["source"]
                for cell in source_notebook["cells"] if cell["cell_type"] == "code"]

def unique_v1_cell(start, must_contain):
    matches = [src for src in source_cells if src.lstrip().startswith(start) and must_contain in src]
    if len(matches) != 1:
        raise ValueError(f"Expected one v1 code cell starting {start!r}; found {len(matches)}")
    return matches[0]

settings_source = unique_v1_cell("# Selection settings:", "corems_config = CoreMSConfig(")
original_functions_source = unique_v1_cell("def json_text(value):", "def run_three_level_workflow(")
options_source = unique_v1_cell("options = {", "result = run_three_level_workflow(")
if ('"revision": 1' not in original_functions_source or
        'PUBLIC_SCHEMA_VERSION = "corems-three-levels-public-1.0"' not in original_functions_source):
    raise ValueError("Select the uploaded v1 simple-NetCDF notebook in Section 1")

functions_source = original_functions_source

V1_CODE_SHA256 = hashlib.sha256((settings_source + "\n" + original_functions_source + "\n" + options_source).encode()).hexdigest()
MODULE_SHA256 = hashlib.sha256(module_file.read_bytes()).hexdigest()
exec(compile(settings_source, str(SingleFile_NOTEBOOK) + ":settings", "exec"), globals())
exec(compile(functions_source, str(SingleFile_NOTEBOOK) + ":definitions", "exec"), globals())

# Parse v1's single-run cell, then execute just its options dictionary assignment.
single_run_tree = ast.parse(options_source)
assignment = single_run_tree.body[0]
if (not isinstance(assignment, ast.Assign) or len(assignment.targets) != 1 or
        not isinstance(assignment.targets[0], ast.Name) or assignment.targets[0].id != "options"):
    raise ValueError("The v1 run cell no longer starts with its options dictionary")
exec(compile(ast.Module(body=[assignment], type_ignores=[]), str(SingleFile_NOTEBOOK) + ":options_only", "exec"), globals())
BASE_OPTIONS = copy.deepcopy(options)
del options
print("Loaded v1 settings and functions:", SingleFile_NOTEBOOK)
print("Source SHA-256:", V1_CODE_SHA256)
print("CoreMS module:", module_file)

Loaded v1 settings and functions: C:\Users\nius206\Documents\ARM_nanodesi\01_process_single_raw_file.ipynb
Source SHA-256: 838cd77f62f2cc2e18ec36b1ae88a19c775d350473b51ede187128546cc80c4b
CoreMS module: C:\Users\nius206\Documents\ARM_nanodesi\corems_topn_processing.py


## 3. Validate Excel rows

Use the 16 headers from the supplied workbook. RAW basenames may omit .raw. The background-folder header also accepts the corrected spelling Background File Location. An optional Output Location column overrides DEFAULT_OUTPUT_ROOT. Field metadata values may be blank.
Altitude values must be finite numbers in meters. Date/time values must use
YYYY-MM-DD and UTC HH:MM or HH:MM:SS. Formatted Excel date/time cells also work.
Validation rejects an end datetime earlier than its start when all four values exist.
Metadata-only mode does not require the sample/background RAW files to remain available.

Min. Altitude must not exceed Max. Altitude. New File Name must be a safe basename with a valid date/time, without an a1/b1/c1 code. Duplicate names within an output root are rejected.

The process step refuses to start any RAW while preflight errors remain. Duplicate New File Name values must be corrected in Excel.


In [11]:
REQUIRED_COLUMNS = (
    "raw_file_name", "raw_file_location", "output_location",
    "background_mode", "background_raw_file", "New File Name",
) + tuple(FIELD_METADATA_COLUMNS)
VALID_MODES = {"same_file", "external_file", "none"}

def excel_path(value):
    value = str(value).strip()
    path = Path(value).expanduser()
    return (path if path.is_absolute() else JOBS_XLSX.parent / path).resolve()

def check_shared_settings():
    validate_settings(BASE_OPTIONS)
    validate_background_settings(BASE_OPTIONS)
    if corems_config.recalibration_enabled:
        reference = Path(corems_config.reference_mass_list_path).expanduser()
        if not reference.is_file():
            raise FileNotFoundError(f"Fix REFERENCE_MASS_LIST_PATH in v1: {reference}")
    url = make_url(corems_config.formula_database_url)
    if url.get_backend_name() == "sqlite" and url.database not in (None, "", ":memory:"):
        database = Path(url.database).expanduser()
        if not database.is_file():
            raise FileNotFoundError(f"Fix FORMULA_DATABASE_URL in v1: {database}")

def read_jobs():
    try:
        table = pd.read_excel(JOBS_XLSX, sheet_name=JOBS_SHEET, dtype=str, keep_default_na=False)
    except ImportError as exc:
        raise ImportError("Install openpyxl in this notebook's CoreMS environment to read .xlsx") from exc
    table.columns = [str(column).strip() for column in table.columns]
    aliases = {
        "Raw File Name": "raw_file_name", "Raw File Location": "raw_file_location",
        "Background Mode": "background_mode", "Background File": "background_raw_file",
        "Backgorund File Location": "background_raw_location",
        "Background File Location": "background_raw_location",
        "Output Location": "output_location",
    }
    table = table.rename(columns=aliases)
    if table.columns.duplicated().any():
        raise ValueError("Duplicate or equivalent column headers in Excel")
    # Ignore entirely blank rows before adding the configured output folder.
    table = table.loc[table.apply(lambda row: any(str(v).strip() for v in row), axis=1)]
    if "output_location" not in table.columns:
        table["output_location"] = str(DEFAULT_OUTPUT_ROOT)
    missing = [name for name in REQUIRED_COLUMNS if name not in table.columns]
    if missing:
        raise ValueError(f"Jobs sheet is missing columns: {missing}")
    jobs = []
    for row_index, record in table.iterrows():
        excel_row = int(row_index) + 2
        values = {name: str(record[name]).strip() for name in REQUIRED_COLUMNS}
        if not any(values.values()):
            continue
        job = {**values, "excel_row": excel_row, "errors": [],
               "sample_path": None, "background_path": None, "output_root": None}
        try:
            job["field_metadata"] = normalized_field_metadata(
                {attr: values[column] for column, attr in FIELD_METADATA_COLUMNS.items()})
        except (TypeError, ValueError) as exc:
            job["field_metadata"] = {}
            job["errors"].append(f"Field metadata: {exc}")
        try:
            job["new_file_name"] = normalize_output_stem(values["New File Name"])
        except ValueError as exc:
            job["new_file_name"] = ""
            job["errors"].append(str(exc))
        name = values["raw_file_name"]
        if name and not Path(name).suffix:
            name += ".raw"
        job["raw_file_name"] = name
        if not name or Path(name).name != name or Path(name).suffix.lower() != ".raw":
            job["errors"].append("raw_file_name must be a .raw basename (no folder)")
        if not values["raw_file_location"]:
            job["errors"].append("raw_file_location is required")
        elif name and Path(name).name == name:
            directory = excel_path(values["raw_file_location"])
            job["sample_path"] = (directory / name).resolve()
            if RUN_MODE == "process" and not directory.is_dir():
                job["errors"].append(f"Sample RAW folder does not exist: {directory}")
            elif RUN_MODE == "process" and not job["sample_path"].is_file():
                job["errors"].append(f"Sample RAW does not exist: {job['sample_path']}")
        if not values["output_location"]:
            job["errors"].append("output_location is required")
        else:
            job["output_root"] = excel_path(values["output_location"])
            if job["output_root"].exists() and not job["output_root"].is_dir():
                job["errors"].append(f"Output location is not a folder: {job['output_root']}")
        mode = values["background_mode"].lower()
        job["background_mode"] = mode
        if mode not in VALID_MODES:
            job["errors"].append("background_mode must be same_file, external_file, or none")
        bg_name = values["background_raw_file"]
        if bg_name and not Path(bg_name).suffix:
            bg_name += ".raw"
        bg_location = str(record.get("background_raw_location", "")).strip()
        if bg_name and bg_location and not Path(bg_name).is_absolute():
            bg_name = str(excel_path(bg_location) / bg_name)
        job["background_raw_file"] = bg_name
        if mode == "external_file":
            if not bg_name:
                job["errors"].append("external_file requires the full background_raw_file path")
            else:
                job["background_path"] = excel_path(bg_name)
                if job["background_path"].suffix.lower() != ".raw" or (RUN_MODE == "process" and not job["background_path"].is_file()):
                    job["errors"].append(f"Background RAW does not exist: {job['background_path']}")
                if job["sample_path"] is not None and job["background_path"] == job["sample_path"]:
                    job["errors"].append("External background must be a different RAW")
        elif mode in {"same_file", "none"} and bg_name:
            job["errors"].append(f"Leave background_raw_file blank for {mode}")
        jobs.append(job)

    seen = {}
    for job in jobs:
        path = job["sample_path"]
        if path is None:
            continue
        key = str(path).casefold()
        if key in seen:
            job["errors"].append(f"Same sample RAW already appears at Excel row {seen[key]['excel_row']}")
            seen[key]["errors"].append(f"Same sample RAW also appears at Excel row {job['excel_row']}")
        else:
            seen[key] = job
    seen_names = {}
    for job in jobs:
        if not job.get("new_file_name") or job["output_root"] is None:
            continue
        key = (str(job["output_root"]).casefold(), job["new_file_name"].casefold())
        if key in seen_names:
            other = seen_names[key]
            message = f"Duplicate New File Name at Excel rows {other['excel_row']} and {job['excel_row']}"
            other["errors"].append(message)
            job["errors"].append(message)
        else:
            seen_names[key] = job
    return jobs

if RUN_MODE not in {"metadata_only", "process"}:
    raise ValueError("RUN_MODE must be metadata_only or process")
if RUN_MODE == "process":
    check_shared_settings()
JOBS = read_jobs()
if ONLY_EXCEL_ROWS is not None:
    requested_rows = set(ONLY_EXCEL_ROWS)
    if not requested_rows or any(not isinstance(row, int) or row < 2 for row in requested_rows):
        raise ValueError("ONLY_EXCEL_ROWS must contain Excel row numbers >= 2, or be None")
    available_rows = {job["excel_row"] for job in JOBS}
    if requested_rows - available_rows:
        raise ValueError(f"Selected Excel rows not found: {sorted(requested_rows - available_rows)}")
    JOBS = [job for job in JOBS if job["excel_row"] in requested_rows]
    print("Selected Excel rows:", sorted(requested_rows))
preflight = pd.DataFrame([
    {"excel_row": j["excel_row"], "raw_file_name": j["raw_file_name"],
     "background_mode": j["background_mode"], "output_location": j["output_location"],
     "status": "invalid" if j["errors"] else "ready", "error": "; ".join(j["errors"])}
    for j in JOBS
])
print(f"Excel rows: {len(JOBS)} | ready: {sum(not j['errors'] for j in JOBS)} | invalid: {sum(bool(j['errors']) for j in JOBS)}")
display(preflight)

Excel rows: 21 | ready: 21 | invalid: 0


,excel_row,raw_file_name,background_mode,output_location,status,error
0,2,10212022_GV_nanoDESI_Neg_T7_SGPCF_041722_D_A_1...,external_file,C:\Users\nius206\Documents\ARM_nanodesi\batch2,ready,
1,3,11142022_GV_nanoDESI_Neg_T7_SGPCF_041822_D_A_3...,external_file,C:\Users\nius206\Documents\ARM_nanodesi\batch2,ready,
2,4,11142022_GV_nanoDESI_Neg_T7_SGPCF_042622_D_A_0...,external_file,C:\Users\nius206\Documents\ARM_nanodesi\batch2,ready,
3,5,20231122_Goldstein_nanoDESI_Neg_GV_20230910_FL...,external_file,C:\Users\nius206\Documents\ARM_nanodesi\batch2,ready,
4,6,20231122_Goldstein_nanoDESI_Neg_GV_20230913_FL...,external_file,C:\Users\nius206\Documents\ARM_nanodesi\batch2,ready,
5,7,20231122_Goldstein_nanoDESI_Neg_GV_20230913_FL...,external_file,C:\Users\nius206\Documents\ARM_nanodesi\batch2,ready,
6,8,20231122_Goldstein_nanoDESI_Neg_GV_20230914_FL...,external_file,C:\Users\nius206\Documents\ARM_nanodesi\batch2,ready,
7,9,20231122_Goldstein_nanoDESI_Neg_GV_20230915_FL...,external_file,C:\Users\nius206\Documents\ARM_nanodesi\batch2,ready,
8,10,20231122_Goldstein_nanoDESI_Neg_GV_20230916_FL...,external_file,C:\Users\nius206\Documents\ARM_nanodesi\batch2,ready,
9,11,20231122_Goldstein_nanoDESI_Neg_GV_SGP_2023071...,external_file,C:\Users\nius206\Documents\ARM_nanodesi\batch2,ready,


## 4. Run the selected mode

`metadata_only` refreshes successful outputs listed in the original ledger and writes a separate metadata update report. It does not change ledger fingerprints or rerun RAW processing. It replaces the legacy user_note attribute with field_note.

`process` runs the existing scientific workflow. Field metadata and New File Name edits are excluded from its scientific fingerprint. Updated code still changes the fingerprint, so use metadata_only when updating results from the earlier notebook version.

Renaming checks all completed targets before moving any files, refuses overwrites, and updates the status file used by the ledger. Run folders and diagnostic filenames stay the same. If c1 was skipped, no c1 file is created.


In [12]:
from contextlib import redirect_stdout, redirect_stderr
REPORT_DIR.mkdir(parents=True, exist_ok=True)
report_key = hashlib.sha256(str(LEDGER_IDENTITY_XLSX.expanduser().resolve()).encode()).hexdigest()[:10]
LEDGER_PATH = REPORT_DIR / f"batch_v1_ledger_{report_key}.json"
SUMMARY_PATH = REPORT_DIR / f"batch_v1_summary_{report_key}.json"

def write_json_atomic(path, value):
    temporary = path.with_name(path.name + ".tmp")
    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
    os.replace(temporary, path)

def read_ledger():
    if not LEDGER_PATH.is_file():
        return {"schema_version": 1, "jobs": {}}
    ledger = json.loads(LEDGER_PATH.read_text(encoding="utf-8"))
    if ledger.get("schema_version") != 1 or not isinstance(ledger.get("jobs"), dict):
        raise ValueError(f"Unknown batch ledger: {LEDGER_PATH}")
    return ledger

def job_key(job):
    payload = str(job["sample_path"]).casefold() + "\0" + str(job["output_root"]).casefold()
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()

def job_options(job):
    settings = copy.deepcopy(BASE_OPTIONS)
    settings["background_mode"] = job["background_mode"]
    settings["background_raw_path"] = (str(job["background_path"])
                                        if job["background_mode"] == "external_file" else None)
    settings["field_metadata"] = job["field_metadata"]
    settings["new_file_name"] = job["new_file_name"]
    return settings

def scientific_fingerprint(job, settings):
    def stamp(path):
        info = Path(path).stat()
        return {"path": str(path), "size": info.st_size, "mtime_ns": info.st_mtime_ns}
    reference = (Path(corems_config.reference_mass_list_path).expanduser().resolve()
                 if corems_config.recalibration_enabled else None)
    scientific_options = {key: value for key, value in settings.items() if key not in {"field_metadata", "new_file_name", "user_note"}}
    payload = {
        "sample": stamp(job["sample_path"]),
        "external_background": stamp(job["background_path"]) if job["background_path"] else None,
        "reference_list": stamp(reference) if reference else None,
        "output_root": str(job["output_root"]),
        "options": scientific_options, "corems_config": asdict(corems_config),
        "v1_code_sha256": V1_CODE_SHA256, "module_sha256": MODULE_SHA256,
        "batch_metadata_patch": "complete-global-metadata-1",
    }
    serialized = json.dumps(payload, sort_keys=True, separators=(",", ":"), default=str)
    return hashlib.sha256(serialized.encode("utf-8")).hexdigest()

def saved_state(run_directory):
    if not run_directory:
        return None
    path = Path(run_directory) / "diagnostics" / "three_level_status.json"
    try:
        return json.loads(path.read_text(encoding="utf-8")) if path.is_file() else None
    except (OSError, ValueError):
        return None

def verified_success(entry, fingerprint):
    if not entry or entry.get("status") != "success" or entry.get("fingerprint") != fingerprint:
        return None
    state = saved_state(entry.get("run_directory"))
    if not state or state.get("status") != "success":
        return None
    for level in ("level1", "level2", "level3"):
        record = state.get("levels", {}).get(level, {})
        if level in ("level1", "level2") and record.get("status") != "success":
            return None
        if level == "level3" and record.get("status") == "skipped":
            continue
        if record.get("status") != "success":
            return None
        path = Path(state.get("netcdf_files", {}).get(level, ""))
        if not path.is_file():
            return None
        try:
            with Dataset(path) as nc:
                if (public_output_level(nc) != level or nc.processing_status != "success" or
                        nc.public_schema_version != PUBLIC_SCHEMA_VERSION):
                    return None
        except (OSError, AttributeError, ValueError):
            return None
    if not (Path(state["run_directory"]) / "data" / "raw_metadata.json").is_file():
        return None
    return state

def sync_metadata(state, field_metadata, new_file_name):
    field_metadata = normalized_field_metadata(field_metadata)
    new_file_name = normalize_output_stem(new_file_name)
    rename_public_level_files(state, new_file_name)
    data_dir = Path(state["run_directory"]) / "data"
    metadata = json.loads((data_dir / "raw_metadata.json").read_text(encoding="utf-8"))
    metadata.pop("user_note", None)
    metadata.update(field_metadata)
    metadata = normalize_public_metadata(metadata)
    # Reconcile late-known counts from the completed processing state.
    for state_name, attr_name in [("n_sample_scans", "n_sample_scans_averaged"),
                                  ("n_background_scans", "n_background_scans_averaged")]:
        if state.get(state_name) is not None:
            metadata[attr_name] = state[state_name]
    # 01 removes obsolete public attributes and groups the tracer attributes.
    update_completed_sample_metadata(state, metadata, data_dir)
    config_path = Path(state["run_directory"]) / "diagnostics" / "three_level_config.json"
    if config_path.is_file():
        config = json.loads(config_path.read_text(encoding="utf-8"))
        config.pop("user_note", None)
        config["field_metadata"] = field_metadata
        config["new_file_name"] = new_file_name
        write_json_atomic(config_path, config)

def nearby_runs(job):
    parent = job["output_root"] / topn.safe_output_key(job["sample_path"].stem)
    return {path.resolve() for path in parent.iterdir() if path.is_dir()} if parent.is_dir() else set()

def summary_row(job, status, state=None, error=""):
    levels = state.get("levels", {}) if state else {}
    level2 = levels.get("level2", {})
    level3 = levels.get("level3", {})
    after_background = level3.get("n_unambiguous_after_background")
    before = level3.get("n_unambiguous_before_cleaning")
    after_oxygen = level3.get("n_unambiguous_after_oxygen")
    after_dbe = level3.get("n_unambiguous_after_dbe_o")
    run_dir = state.get("run_directory") if state else None
    metadata_path = Path(run_dir) / "data" / "raw_metadata.json" if run_dir else None
    try:
        metadata = json.loads(metadata_path.read_text(encoding="utf-8")) if metadata_path and metadata_path.is_file() else {}
    except (OSError, ValueError):
        metadata = {}
    return {
        "excel_row": job["excel_row"], "raw_file_name": job["raw_file_name"],
        "sample_raw_path": str(job["sample_path"] or job["raw_file_location"]),
        "output_location": str(job["output_root"] or job["output_location"]),
        "background_mode": job["background_mode"],
        "background_raw_file": str(job["background_path"] or job["background_raw_file"]),
        "New File Name": job.get("new_file_name", ""),
        **{column: job.get(column, "") for column in FIELD_METADATA_COLUMNS}, "status": status,
        "failed_level": state.get("failed_level") if state else None,
        "error": error or (state.get("error") if state else ""),
        "run_directory": run_dir,
        "level1_nc": state.get("netcdf_files", {}).get("level1") if state else None,
        "level2_nc": state.get("netcdf_files", {}).get("level2") if state else None,
        "level3_nc": state.get("netcdf_files", {}).get("level3") if state else None,
        "level1_status": levels.get("level1", {}).get("status"),
        "level2_status": level2.get("status"), "level3_status": level3.get("status"),
        "level3_skip_reason": level3.get("reason"),
        "selected_tracer_name": metadata.get("selected_tracer_name"),
        "n_sample_scans": state.get("n_sample_scans") if state else None,
        "n_background_scans": state.get("n_background_scans") if state else None,
        "n_total_peaks": level2.get("n_total_peaks"),
        "n_unambiguous_peaks": level2.get("n_unambiguous_peaks"),
        "n_ambiguous_peaks": level2.get("n_ambiguous_peaks"),
        "n_unassigned_peaks": level2.get("n_unassigned_peaks"),
        "n_removed_by_background": before - after_background if isinstance(before, int) and isinstance(after_background, int) else None,
        "n_removed_without_oxygen": level3.get("n_removed_without_oxygen"),
        "n_removed_by_dbe_o": after_oxygen - after_dbe if isinstance(after_oxygen, int) and isinstance(after_dbe, int) else None,
        "n_final_unambiguous": after_dbe,
    }

def run_batch(jobs):
    invalid = [job for job in jobs if job["errors"]]
    if invalid:
        details = "\n".join(f"Excel row {j['excel_row']}: {'; '.join(j['errors'])}" for j in invalid)
        raise ValueError("Fix all preflight errors before processing RAW files:\n" + details)
    ledger = read_ledger()
    rows = []
    for index, job in enumerate(jobs, 1):
        label = f"Excel row {job['excel_row']} ({job['raw_file_name']})"
        if job["errors"]:
            row = summary_row(job, "invalid", error="; ".join(job["errors"]))
        else:
            key = job_key(job)
            fingerprint = None
            started = False
            before = set()
            try:
                settings = job_options(job)
                fingerprint = scientific_fingerprint(job, settings)
                previous = ledger["jobs"].get(key)
                known = None if job["excel_row"] in FORCE_RERUN_EXCEL_ROWS else verified_success(previous, fingerprint)
                if known is not None:
                    sync_metadata(known, job["field_metadata"], job["new_file_name"])
                    row = summary_row(job, "already_done", state=known)
                else:
                    job["output_root"].mkdir(parents=True, exist_ok=True)
                    before = nearby_runs(job)
                    ledger["jobs"][key] = {
                        "status": "running", "fingerprint": fingerprint,
                        "v1_code_sha256": V1_CODE_SHA256,
                        "started_at_utc": datetime.now(timezone.utc).isoformat(),
                    }
                    write_json_atomic(LEDGER_PATH, ledger)
                    started = True
                    print(f"[{index}/{len(jobs)}] Processing {label}...", flush=True)
                    log_path = REPORT_DIR / f"batch_v1_row_{job['excel_row']:04d}.log"
                    with log_path.open("w", encoding="utf-8") as log, \
                        redirect_stdout(log), redirect_stderr(log):
                        state = run_three_level_workflow(
                            job["sample_path"], job["output_root"], corems_config, settings
                        )
                    row = summary_row(job, "success", state=state)
            except Exception as exc:
                state = None
                if started:
                    try:
                        fresh = nearby_runs(job) - before
                        latest = max(fresh, key=lambda path: path.stat().st_mtime_ns) if fresh else None
                        state = saved_state(latest) if latest else None
                    except (OSError, ValueError):
                        pass
                row = summary_row(job, "failed", state=state, error=f"{type(exc).__name__}: {exc}")
                print(f"[{index}/{len(jobs)}] {label}: FAILED ({row['error']})")
                traceback.print_exc(limit=3)
            # Retain an earlier successful ledger entry if updating only its metadata fails.
            # The next batch pass can retry the metadata without repeating CoreMS.
            if row["status"] == "success" or (row["status"] == "failed" and started):
                ledger["jobs"][key] = {
                    "status": row["status"], "fingerprint": fingerprint,
                    "run_directory": row["run_directory"], "error": row["error"],
                    "v1_code_sha256": V1_CODE_SHA256,
                    "updated_at_utc": datetime.now(timezone.utc).isoformat(),
                }
                write_json_atomic(LEDGER_PATH, ledger)
        rows.append(row)
        write_json_atomic(SUMMARY_PATH, {
            "jobs_xlsx": str(JOBS_XLSX), "SingleFileNotebook": str(SingleFile_NOTEBOOK),
            "v1_code_sha256": V1_CODE_SHA256,
            "updated_at_utc": datetime.now(timezone.utc).isoformat(),
            "rows": rows,
        })
        if row["status"] != "failed":
            print(f"[{index}/{len(jobs)}] {label}: {row['status']} | Level 3: {row['level3_status']}")
    return pd.DataFrame(rows)

def update_batch_metadata(jobs):
    """Use successful ledger entries only. Never call scientific processing."""
    ledger = read_ledger()
    rows = []
    for job in jobs:
        state = None
        try:
            if job["errors"]:
                raise ValueError("; ".join(job["errors"]))
            entry = ledger["jobs"].get(job_key(job))
            state = verified_success(entry, entry.get("fingerprint") if entry else None)
            if state is None:
                raise ValueError("No verified successful run in the selected ledger. Check REPORT_DIR and LEDGER_IDENTITY_XLSX.")
            # Protect against a mismatched or stale ledger path.
            for path_string in state.get("netcdf_files", {}).values():
                with Dataset(path_string) as nc:
                    if nc.sample_raw_filename.casefold() != job["raw_file_name"].casefold():
                        raise ValueError("Saved NetCDF sample filename does not match this Excel row")
            sync_metadata(state, job["field_metadata"], job["new_file_name"])
            row = summary_row(job, "metadata_updated", state=state)
        except Exception as exc:
            row = summary_row(job, "metadata_not_updated", state=state, error=str(exc))
        rows.append(row)
        print(f"Excel row {job['excel_row']}: {row['status']} {row['error']}")
    target = REPORT_DIR / f"metadata_update_summary_{report_key}.json"
    write_json_atomic(target, {"jobs_xlsx": str(JOBS_XLSX), "rows": rows})
    print("Metadata update summary:", target)
    return pd.DataFrame(rows)


results = update_batch_metadata(JOBS) if RUN_MODE == "metadata_only" else run_batch(JOBS)
display(results)
print("Summary:", SUMMARY_PATH)
print("Progress ledger:", LEDGER_PATH)

[1/21] Processing Excel row 2 (10212022_GV_nanoDESI_Neg_T7_SGPCF_041722_D_A_1_1.raw)...


[1/21] Excel row 2 (10212022_GV_nanoDESI_Neg_T7_SGPCF_041722_D_A_1_1.raw): success | Level 3: success
[2/21] Processing Excel row 3 (11142022_GV_nanoDESI_Neg_T7_SGPCF_041822_D_A_3_4_5_01.raw)...
[2/21] Excel row 3 (11142022_GV_nanoDESI_Neg_T7_SGPCF_041822_D_A_3_4_5_01.raw): success | Level 3: success
[3/21] Processing Excel row 4 (11142022_GV_nanoDESI_Neg_T7_SGPCF_042622_D_A_01.raw)...
[3/21] Excel row 4 (11142022_GV_nanoDESI_Neg_T7_SGPCF_042622_D_A_01.raw): success | Level 3: success
[4/21] Processing Excel row 5 (20231122_Goldstein_nanoDESI_Neg_GV_20230910_FLIGHT2_01.raw)...
[4/21] Excel row 5 (20231122_Goldstein_nanoDESI_Neg_GV_20230910_FLIGHT2_01.raw): success | Level 3: success
[5/21] Processing Excel row 6 (20231122_Goldstein_nanoDESI_Neg_GV_20230913_FLIGHT2_01.raw)...
[5/21] Excel row 6 (20231122_Goldstein_nanoDESI_Neg_GV_20230913_FLIGHT2_01.raw): success | Level 3: success
[6/21] Processing Excel row 7 (20231122_Goldstein_nanoDESI_Neg_GV_20230913_FLIGHT3_01.raw)...
[6/21] Excel

,excel_row,raw_file_name,sample_raw_path,output_location,background_mode,background_raw_file,New File Name,Location,Site,System,...,n_sample_scans,n_background_scans,n_total_peaks,n_unambiguous_peaks,n_ambiguous_peaks,n_unassigned_peaks,n_removed_by_background,n_removed_without_oxygen,n_removed_by_dbe_o,n_final_unambiguous
0,2,10212022_GV_nanoDESI_Neg_T7_SGPCF_041722_D_A_1...,D:\TAP_Data\raw_wo_background\Sample_Files\102...,C:\Users\nius206\Documents\ARM_nanodesi\batch2,external_file,D:\TAP_Data\raw_wo_background\Background_Files...,sgphrmssioutasd0m650mCF.20220417.141000,SGP,CF,Sioutas stage D; Teflon,...,50,25,1151,992,9,150,230,0,0,762
1,3,11142022_GV_nanoDESI_Neg_T7_SGPCF_041822_D_A_3...,D:\TAP_Data\raw_wo_background\Sample_Files\111...,C:\Users\nius206\Documents\ARM_nanodesi\batch2,external_file,D:\TAP_Data\raw_wo_background\Background_Files...,sgphrmssioutasd0m550mCF.20220418.205700,SGP,CF,Sioutas stage D; Teflon,...,50,25,1702,1297,9,396,392,0,0,905
2,4,11142022_GV_nanoDESI_Neg_T7_SGPCF_042622_D_A_0...,D:\TAP_Data\raw_wo_background\Sample_Files\111...,C:\Users\nius206\Documents\ARM_nanodesi\batch2,external_file,D:\TAP_Data\raw_wo_background\Background_Files...,sgphrmssioutasd0m750mCF.20220426.154000,SGP,CF,Sioutas stage D; Teflon,...,50,25,2212,1675,20,517,439,1,2,1233
3,5,20231122_Goldstein_nanoDESI_Neg_GV_20230910_FL...,D:\TAP_Data\raw_wo_background\Sample_Files\202...,C:\Users\nius206\Documents\ARM_nanodesi\batch2,external_file,D:\TAP_Data\raw_wo_background\Background_Files...,sgphrmssioutasd0m500mCF.20230910.192700,SGP,CF,Sioutas stage D; Teflon,...,50,25,997,894,9,94,38,0,2,854
4,6,20231122_Goldstein_nanoDESI_Neg_GV_20230913_FL...,D:\TAP_Data\raw_wo_background\Sample_Files\202...,C:\Users\nius206\Documents\ARM_nanodesi\batch2,external_file,D:\TAP_Data\raw_wo_background\Background_Files...,sgphrmssioutasd0m1000mCF.20230913.195900,SGP,CF,Sioutas stage D; Teflon,...,50,25,1686,1113,7,566,51,6,23,1033
5,7,20231122_Goldstein_nanoDESI_Neg_GV_20230913_FL...,D:\TAP_Data\raw_wo_background\Sample_Files\202...,C:\Users\nius206\Documents\ARM_nanodesi\batch2,external_file,D:\TAP_Data\raw_wo_background\Background_Files...,sgphrmssioutasd0m1000mCF.20230913.233600,SGP,CF,Sioutas stage D; Teflon,...,50,25,1439,973,10,456,53,6,18,896
6,8,20231122_Goldstein_nanoDESI_Neg_GV_20230914_FL...,D:\TAP_Data\raw_wo_background\Sample_Files\202...,C:\Users\nius206\Documents\ARM_nanodesi\batch2,external_file,D:\TAP_Data\raw_wo_background\Background_Files...,sgphrmssioutasd0m385mCF.20230914.040900,SGP,CF,Sioutas stage D; Teflon,...,50,25,1336,1022,24,290,47,2,0,973
7,9,20231122_Goldstein_nanoDESI_Neg_GV_20230915_FL...,D:\TAP_Data\raw_wo_background\Sample_Files\202...,C:\Users\nius206\Documents\ARM_nanodesi\batch2,external_file,D:\TAP_Data\raw_wo_background\Background_Files...,sgphrmssioutasd0m500mCF.20230915.020600,SGP,CF,Sioutas stage D; Teflon,...,50,25,1797,1187,34,576,62,12,4,1109
8,10,20231122_Goldstein_nanoDESI_Neg_GV_20230916_FL...,D:\TAP_Data\raw_wo_background\Sample_Files\202...,C:\Users\nius206\Documents\ARM_nanodesi\batch2,external_file,D:\TAP_Data\raw_wo_background\Background_Files...,sgphrmssioutasd0m1350mCF.20230916.204100,SGP,CF,Sioutas stage D; Teflon,...,50,25,1633,1209,11,413,52,0,35,1122
9,11,20231122_Goldstein_nanoDESI_Neg_GV_SGP_2023071...,D:\TAP_Data\raw_wo_background\Sample_Files\202...,C:\Users\nius206\Documents\ARM_nanodesi\batch2,external_file,D:\TAP_Data\raw_wo_background\Background_Files...,sgphrmssioutasd0m1015mCF.20230713.145900,SGP,CF,Sioutas stage D; Teflon,...,50,25,2001,1097,0,904,64,17,22,994


Summary: C:\Users\nius206\Documents\ARM_nanodesi\named_output_reports\batch_v1_summary_fbca81d81d.json
Progress ledger: C:\Users\nius206\Documents\ARM_nanodesi\named_output_reports\batch_v1_ledger_fbca81d81d.json
